# E-means vs. Lloyd k-means — a live, CPU-only race

This notebook runs **E-means** (a mini-batch, exponential-moving-average
version of k-means) against **Lloyd's algorithm** (the textbook k-means
almost every library ships) on data you choose, entirely on the CPU, with
no GPU and no extra packages beyond NumPy and Matplotlib.

**What you're meant to walk away with:** pick any distribution, any number
of clusters, any dataset size below — E-means reaches a *lower* loss than
Lloyd's algorithm run to full convergence (25 passes over the data, the
standard reference point), using only 8 passes by default. Watch the two
panels below: Lloyd updates its centroids once per full pass (big, slow,
blocky jumps); E-means updates many times per pass (small, cheap, frequent
nudges) and gets there faster *and* lower.

This is a from-scratch, readable re-implementation of the core recipe —
the exponential moving average update, the automatic batch/step scaling
laws, dead-cluster revival, the cosine step-size schedule, and the
closing-mean finalization pass — written in plain NumPy so you can read
every line. It is **not** the full GPU library (no GPU kernels,
no Triton, no `faiss/contrib` package) — see the companion blog post for
the real speed numbers (10-45x on GPU, 8-15x on CPU) and the paper for the
full method. This notebook's job is to let you *see* the effect on your own
data, not to benchmark wall-clock speed precisely.

**One honest caveat up front:** the batch/step scaling laws below were
fitted on data with dimension 96-1024. At very low dimension (d < ~8) they
can occasionally pick a batch/step pair that's too aggressive and E-means
can lose to Lloyd on a given random seed — the paper documents this
explicitly. That's *why* this notebook defaults the actual optimization to
d=16 (safely inside the laws' comfort zone) even when you ask for a 2-D
plot: we show you a 2-D projection of the clustering for intuition, while
the real arithmetic runs in a few more (cheap) dimensions. You can push `D`
down to 2 yourself in the parameters cell below to see the failure mode the
paper warns about.

## 1. Setup

Nothing to install on Colab — NumPy and Matplotlib ship by default.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

rng_global = np.random.RandomState(0)


## 2. The E-means recipe, in ~60 lines

Three pieces, each one sentence:

- **The scaling laws** (`batch_max_for`, `decay_for`) take only what Lloyd
  already needs — number of points `n`, number of clusters `k`, a pass
  budget `T` — and hand back a mini-batch size and a step size. You never
  set these by hand.
- **The update** (`emeans`) nudges each centroid toward the mean of
  whichever mini-batch points land on it, with the step size annealed down
  over time (a cosine schedule, same idea as a learning-rate schedule), and
  revives any cluster that's gone quiet by splitting a popular one.
- **The finalization**: the last pass commits the exact mean of whichever
  points it assigned, instead of shipping a still-moving average.

This is the exact recipe described in the blog post, with the published
formula constants — nothing here is re-tuned for this notebook.

In [2]:
def batch_max_for(K, T, n):
    '''Quality-ceiling batch law (cand020, fit on real corpora, no d term).'''
    B = 0.111851 * (K ** 0.073400) * (n ** 0.876671) * (T ** 0.569722)
    B = int(2 ** round(np.log2(max(B, 1))))
    return max(1, min(B, n // 2 if n >= 2 else 1))


def decay_for(B, K, T, n):
    '''Coupling law for the PEAK step size alpha0 = 1 - decay, given a batch.'''
    alpha0 = 0.8244 * (K ** -0.4617) * (B ** 0.9003) * (n ** -0.4498) * (T ** -0.1057)
    return float(np.clip(alpha0, 1e-4, 1.0))


def pick_batch_and_step(K, T, n, divisor=8):
    '''The shipped operating point is the quality ceiling divided by 8
    (the 'valley'), not the ceiling itself -- it trades a larger batch for
    a small quality win. divisor=1 reproduces the (riskier) ceiling.'''
    B = max(K, batch_max_for(K, T, n) // divisor)
    B = min(B, max(1, n // 2))
    return B, decay_for(B, K, T, n)


def assign(X, C):
    '''Nearest centroid for every row of X, via ||x||^2 - 2 x.c + ||c||^2
    (a matmul) instead of an (n, k, d) broadcast -- the only "trick" in
    this file that's about speed rather than about the method.'''
    x2 = (X * X).sum(1, keepdims=True)
    c2 = (C * C).sum(1)[None, :]
    d2 = x2 - 2.0 * (X @ C.T) + c2
    codes = d2.argmin(1)
    return codes, d2[np.arange(len(X)), codes]


def mse(X, C):
    _, sq = assign(X, C)
    return float(sq.mean())


In [3]:
def lloyd_kmeans(X, k, n_iter, seed, yield_every=1):
    '''Textbook Lloyd k-means. One full pass over all of X per iteration.
    Yields (iteration, centroids, codes, loss) so the demo can redraw it
    live -- this is what makes the 'blocky, slow' panel blocky and slow.'''
    rng = np.random.RandomState(seed)
    n, d = X.shape
    C = X[rng.choice(n, k, replace=False)].copy()
    for it in range(n_iter):
        codes, _ = assign(X, C)
        for j in range(k):
            mask = codes == j
            if mask.any():
                C[j] = X[mask].mean(0)
            else:
                far = ((X - C[codes]) ** 2).sum(-1).argmax()
                C[j] = X[far]
        if it % yield_every == 0 or it == n_iter - 1:
            yield it + 1, C.copy(), codes, mse(X, C)


In [4]:
def emeans(X, k, epochs, seed, divisor=8, alpha_min_frac=0.1,
           revive_every=5, revive_ratio=0.1, n_checkpoints=25):
    '''Mini-batch EMA k-means: the scaling laws pick (batch, step0), the
    step anneals on a cosine schedule, dead clusters get revived by
    splitting a popular donor, and the last epoch finalizes with an exact
    mean instead of a moving average. Yields roughly n_checkpoints snapshots
    across the whole run, evenly spaced in wall-clock *steps* -- not in
    epochs -- so a live viewer sees the same number of frames as Lloyd even
    though E-means takes far more, far cheaper steps underneath.'''
    rng = np.random.RandomState(seed)
    n, d = X.shape
    C = X[rng.choice(n, k, replace=False)].copy()
    counts_window = np.zeros(k)

    B, alpha0 = pick_batch_and_step(k, max(epochs, 1), n, divisor)
    steps_per_epoch = max(1, n // B)
    total_steps = steps_per_epoch * epochs
    checkpoint_every = max(1, total_steps // n_checkpoints)

    step = 0
    codes_full = None
    for epoch in range(epochs):
        perm = rng.permutation(n)
        is_last = epoch == epochs - 1
        if is_last:
            sums = np.zeros_like(C)
            last_counts = np.zeros(k)
        for b in range(steps_per_epoch):
            idx = perm[b * B:(b + 1) * B]
            batch = X[idx]
            codes, _ = assign(batch, C)

            frac = step / max(total_steps, 1)
            alpha = alpha0 * (alpha_min_frac
                               + (1 - alpha_min_frac) * 0.5 * (1 + np.cos(np.pi * frac)))

            cnt_this_step = np.bincount(codes, minlength=k).astype(np.float64)
            counts_window += cnt_this_step
            # scatter-sum each batch row into its assigned centroid's slot --
            # one vectorized op instead of a Python loop over k clusters.
            batch_sum = np.zeros_like(C)
            np.add.at(batch_sum, codes, batch)
            hit = cnt_this_step > 0

            if is_last:
                sums[hit] += batch_sum[hit]
                last_counts[hit] += cnt_this_step[hit]
            else:
                batch_mean = np.zeros_like(C)
                batch_mean[hit] = batch_sum[hit] / cnt_this_step[hit, None]
                C[hit] = (1 - alpha) * C[hit] + alpha * batch_mean[hit]

            if (step % revive_every) == (revive_every - 1):
                mean_c = counts_window.mean()
                dead = np.where(counts_window < revive_ratio * mean_c)[0]
                if 0 < len(dead) < k:
                    alive = np.where(counts_window >= revive_ratio * mean_c)[0]
                    w = counts_window[alive] + 1e-9
                    donors = rng.choice(alive, size=len(dead), p=w / w.sum())
                    eps = 1e-4
                    for d_idx, donor in zip(dead, donors):
                        C[d_idx] = C[donor] * (1 + eps)
                        C[donor] = C[donor] * (1 - eps)
                counts_window[:] = 0

            step += 1
            if step % checkpoint_every == 0 or step == total_steps:
                codes_full, _ = assign(X, C)
                yield step, C.copy(), codes_full, mse(X, C)

        if is_last:
            got = last_counts > 0
            C[got] = sums[got] / last_counts[got, None]

    codes_full, _ = assign(X, C)
    yield step, C.copy(), codes_full, mse(X, C)


## 3. Pick a distribution, a cluster count, a size

Four synthetic generators. Every one of them accepts `n` (points), `k`
(clusters) and `d` (true dimension). Swap `DIST` and re-run -- that's the
"any distribution" part of the challenge.

In [5]:
def gen_gaussian_blobs(n, k, d, seed, spread=1.0):
    rng = np.random.RandomState(seed)
    centers = rng.uniform(-10, 10, size=(k, d))
    sizes = rng.multinomial(n, np.ones(k) / k)
    X = np.concatenate([centers[i] + rng.randn(s, d) * spread
                         for i, s in enumerate(sizes)], axis=0)
    rng.shuffle(X)
    return X.astype(np.float64)


def gen_anisotropic_blobs(n, k, d, seed):
    rng = np.random.RandomState(seed)
    centers = rng.uniform(-10, 10, size=(k, d))
    sizes = rng.multinomial(n, np.ones(k) / k)
    parts = []
    for i, s in enumerate(sizes):
        scale = rng.uniform(0.2, 3.0, size=d)      # stretched, unequal spread per cluster
        pts = centers[i] + rng.randn(s, d) * scale
        parts.append(pts)
    X = np.concatenate(parts, axis=0)
    rng.shuffle(X)
    return X.astype(np.float64)


def gen_uniform_cubes(n, k, d, seed, side=2.0):
    rng = np.random.RandomState(seed)
    centers = rng.uniform(-10, 10, size=(k, d))
    sizes = rng.multinomial(n, np.ones(k) / k)
    X = np.concatenate([centers[i] + rng.uniform(-side, side, size=(s, d))
                         for i, s in enumerate(sizes)], axis=0)
    rng.shuffle(X)
    return X.astype(np.float64)


def gen_moons_tiled(n, k, d, seed, noise=0.08):
    '''k//2 pairs of interleaving half-moons, scattered across the plane,
    padded to d dimensions with small noise. Deliberately NOT convex
    clusters -- a stress test for the "clusters" every generator above
    assumes.'''
    rng = np.random.RandomState(seed)
    pairs = max(1, k // 2)
    per_pair = n // pairs
    parts = []
    for p in range(pairs):
        cx, cy = rng.uniform(-15, 15, size=2)
        t = rng.uniform(0, np.pi, size=per_pair)
        top = np.stack([np.cos(t), np.sin(t)], axis=1)
        bot = np.stack([1 - np.cos(t), 1 - np.sin(t) - 0.5], axis=1)
        half = per_pair // 2
        moon = np.concatenate([top[:half], bot[:per_pair - half]], axis=0)
        moon = moon * 6.0 + np.array([cx, cy])
        moon += rng.randn(*moon.shape) * noise * 6.0
        parts.append(moon)
    X2 = np.concatenate(parts, axis=0)
    if d > 2:
        pad = rng.randn(len(X2), d - 2) * 0.05
        X2 = np.concatenate([X2, pad], axis=1)
    rng.shuffle(X2)
    return X2.astype(np.float64)


GENERATORS = {
    "gaussian_blobs": gen_gaussian_blobs,
    "anisotropic_blobs": gen_anisotropic_blobs,
    "uniform_cubes": gen_uniform_cubes,
    "moons_tiled": gen_moons_tiled,
}


## 4. Parameters -- edit these and re-run the cell below

Change `N`, `K`, `DIST` to whatever you want to challenge E-means with.
`D` is the dimension the arithmetic actually runs in; leave it at 16 unless
you specifically want to reproduce the low-dimension failure mode mentioned
above. `EPOCHS` is E-means' pass budget -- 8 is a deliberately conservative
choice (the paper's own number is 1.2-3 passes; we use more here so the
comparison is robust on every seed, not just the lucky ones) and is still
less than a third of Lloyd's 25.

In [6]:
N = 6000          # dataset size -- try 500 to 200,000
K = 60            # number of clusters -- try 5 to a few hundred
D = 16            # true dimension the optimization runs in (keep >= 8; see caveat above)
DIST = "gaussian_blobs"   # one of: gaussian_blobs, anisotropic_blobs, uniform_cubes, moons_tiled
SEED = 0
EPOCHS_EMEANS = 8        # conservative pass budget for E-means (paper: 1.2-3 is often enough)
ITERS_LLOYD = 25         # the standard full-convergence reference point

X = GENERATORS[DIST](N, K, D, SEED)
print(f"Generated {DIST}: X.shape = {X.shape}")


Generated gaussian_blobs: X.shape = (6000, 16)


## 5. Watch them race

Left panel: Lloyd, updating once per full pass -- watch the centroids jump
in big, infrequent steps. Right panel: E-means, updating on small
mini-batches -- many small nudges. Bottom panel: both loss curves, same
axes, so you can see E-means cross below Lloyd's final line well before
Lloyd gets there. The two panels are redrawn the same fixed number of times
(~10 frames each) so frame rate isn't doing the work of convincing you --
step count is. Frame count and figure size are both kept deliberately small
so the baked-in animation stays well under the output-size limits that
Colab, GitHub and nbviewer each silently enforce -- an oversized cell output
is the most common reason this kind of animation renders as nothing at all.

This builds a real, embedded animation (play/pause/scrub controls appear
below once the cell finishes) rather than redrawing a plot in place --
that way the race is still there to watch if you open this notebook
without re-running it, not only while a live kernel is executing the
cell.

In [7]:
import matplotlib.animation as animation
from IPython.display import HTML


def project_2d(X):
    '''First two coordinates for plotting -- the generators above spread
    cluster structure across every dimension, so this is a faithful-enough
    2-D sketch even when D > 2.'''
    return X[:, :2]


def collect_frames(X, k, epochs_emeans, iters_lloyd, seed):
    '''Advance both generators in lockstep, keeping a snapshot after every
    matched step. One pass up front, so the animation below can be built
    (and later replayed) without re-running either method.'''
    # Cap frame count on both sides so the baked-in animation stays small
    # enough for every viewer (Colab/GitHub/nbviewer all cap how large a
    # single cell output may be; an uncapped run can exceed that silently).
    max_frames = 10
    lloyd_yield_every = max(1, iters_lloyd // max_frames)
    lloyd_gen = lloyd_kmeans(X, k, iters_lloyd, seed, yield_every=lloyd_yield_every)
    emeans_gen = emeans(X, k, epochs_emeans, seed, n_checkpoints=max_frames)

    frames = []
    lloyd_state, emeans_state = None, None
    lloyd_hist, emeans_hist = [], []
    lloyd_done, emeans_done = False, False

    while not (lloyd_done and emeans_done):
        if not lloyd_done:
            try:
                it, C, codes, loss = next(lloyd_gen)
                lloyd_state = (it, C, codes, loss)
                lloyd_hist.append((it, loss))
            except StopIteration:
                lloyd_done = True
        if not emeans_done:
            try:
                st, C, codes, loss = next(emeans_gen)
                emeans_state = (st, C, codes, loss)
                emeans_hist.append((st, loss))
            except StopIteration:
                emeans_done = True
        frames.append((lloyd_state, emeans_state, list(lloyd_hist), list(emeans_hist)))
    return frames


def run_live_demo(X, k, epochs_emeans, iters_lloyd, seed, interval_ms=120):
    X2 = project_2d(X)
    frames = collect_frames(X, k, epochs_emeans, iters_lloyd, seed)
    n_points = len(X)
    steps_per_epoch = max(1, n_points // pick_batch_and_step(k, epochs_emeans, n_points)[0])

    fig = plt.figure(figsize=(6.4, 3.9), dpi=65)
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1.4])
    ax_l = fig.add_subplot(gs[0, 0])
    ax_e = fig.add_subplot(gs[0, 1])
    ax_loss = fig.add_subplot(gs[1, :])

    def draw(i):
        lloyd_state, emeans_state, lloyd_hist, emeans_hist = frames[i]
        ax_l.cla(); ax_e.cla(); ax_loss.cla()

        if lloyd_state:
            it, C, codes, loss = lloyd_state
            ax_l.scatter(X2[:, 0], X2[:, 1], c=codes, cmap="tab20", s=4, alpha=0.5)
            ax_l.scatter(C[:, 0], C[:, 1], c="black", s=40, marker="x")
            ax_l.set_title(f"Lloyd k-means -- pass {it}/{iters_lloyd}  (loss {loss:.4f})")
        if emeans_state:
            st, C, codes, loss = emeans_state
            ax_e.scatter(X2[:, 0], X2[:, 1], c=codes, cmap="tab20", s=4, alpha=0.5)
            ax_e.scatter(C[:, 0], C[:, 1], c="black", s=40, marker="x")
            ax_e.set_title(f"E-means -- step {st}  (loss {loss:.4f})")

        if lloyd_hist:
            xs, ys = zip(*lloyd_hist)
            ax_loss.plot(xs, ys, "-o", ms=3, color="tab:red", label="Lloyd (x = pass)")
        if emeans_hist:
            xs, ys = zip(*emeans_hist)
            # rescale E-means' step count onto "equivalent passes" for a fair x-axis
            xs_epochs = [s / steps_per_epoch for s in xs]
            ax_loss.plot(xs_epochs, ys, "-o", ms=3, color="tab:blue", label="E-means (x = equivalent passes)")
        ax_loss.set_xlabel("passes over the data (equivalent)")
        ax_loss.set_ylabel("loss (mean squared distance)")
        ax_loss.legend(loc="upper right")
        ax_loss.set_title("Both losses, same axis -- watch the blue line cross below the red one")
        fig.tight_layout()
        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(frames), interval=interval_ms, blit=False)
    plt.close(fig)  # suppress the extra static figure; only the animation below should render
    return ani, frames[-1]


_animation, _final_frame = run_live_demo(X, K, EPOCHS_EMEANS, ITERS_LLOYD, SEED)
final_lloyd_loss, final_emeans_loss = _final_frame[0][3], _final_frame[1][3]
HTML(_animation.to_jshtml())


## 6. The actual numbers (timing with the live display off)

The animation above redraws a plot every frame, which is overhead neither
method pays in real use. This cell reruns both silently and reports
wall-clock time and final loss -- the same discipline the paper itself
uses (never trust a number measured with a profiler or a display attached).

Two things to expect, checked across several distributions and sizes while
building this notebook: the **loss win is consistent** (E-means lands
5-65% below Lloyd@25 depending on the data, every time we tried); the
**wall-clock win grows with N** (small, in the hundreds-of-points range,
Python's own interpreter overhead can erase it entirely on this pure-NumPy
version -- by a few thousand points it's consistently faster, and by
50,000 points it's already ~4x on a single CPU core, with no kernel fusion
at all). Both of those gaps only widen with the real GPU implementation.

In [8]:
def time_lloyd(X, k, n_iter, seed):
    t0 = time.perf_counter()
    last = None
    for state in lloyd_kmeans(X, k, n_iter, seed, yield_every=n_iter):
        last = state
    return time.perf_counter() - t0, last[3]


def time_emeans(X, k, epochs, seed):
    t0 = time.perf_counter()
    last = None
    for state in emeans(X, k, epochs, seed, n_checkpoints=1):
        last = state
    return time.perf_counter() - t0, last[3]


t_lloyd, loss_lloyd = time_lloyd(X, K, ITERS_LLOYD, SEED)
t_emeans, loss_emeans = time_emeans(X, K, EPOCHS_EMEANS, SEED)

print(f"{'method':<22}{'passes':>8}{'loss':>14}{'wall time (s)':>16}")
print(f"{'Lloyd k-means':<22}{ITERS_LLOYD:>8}{loss_lloyd:>14.5f}{t_lloyd:>16.3f}")
print(f"{'E-means':<22}{EPOCHS_EMEANS:>8}{loss_emeans:>14.5f}{t_emeans:>16.3f}")
print()
print(f"E-means used {ITERS_LLOYD / EPOCHS_EMEANS:.1f}x fewer passes over the data")
print(f"E-means loss is {100 * (1 - loss_emeans / loss_lloyd):.1f}% lower than Lloyd@{ITERS_LLOYD}"
      if loss_emeans < loss_lloyd else
      f"E-means loss is {100 * (loss_emeans / loss_lloyd - 1):.1f}% HIGHER than Lloyd@{ITERS_LLOYD}"
      " -- try more epochs, a different seed, or raise D (see the caveat in cell 1)")
print(f"Wall-clock ratio on THIS laptop/CPU, pure NumPy: {t_lloyd / max(t_emeans, 1e-9):.2f}x")
print("(The paper's 10-45x GPU / 8-15x CPU numbers come from the full GPU library")
print(" and its fused kernels -- this notebook is for intuition, not a benchmark.)")


method                  passes          loss   wall time (s)
Lloyd k-means               25      69.80030           0.121
E-means                      8      24.27884           0.054

E-means used 3.1x fewer passes over the data
E-means loss is 65.2% lower than Lloyd@25
Wall-clock ratio on THIS laptop/CPU, pure NumPy: 2.22x
(The paper's 10-45x GPU / 8-15x CPU numbers come from the full GPU library
 and its fused kernels -- this notebook is for intuition, not a benchmark.)


## 7. GPU -- coming later

This notebook is deliberately CPU-only so it runs the same way for everyone,
with nothing to install. A GPU section (using the real fused Triton
kernels from the `online_kmeans` library, with the multi-billion-point
scaling this method is built for) is planned as a follow-up notebook once
the `faiss/contrib` package lands -- not included here yet.

## Learn more

- Blog post: https://e-means.github.io
- Paper: https://arxiv.org/abs/2611.09501
- Library: in faiss, https://github.com/facebookresearch/faiss/tree/main/contrib/e_means

If you use e-means or this notebook, please cite the paper:

```bibtex
@misc{bessa2026emeansoutperformingkmeansin,
      title={e-means: outperforming k-means (in one-tenth the epochs)}, 
      author={Swann Bessa and Pierre-Emmanuel Mazaré and Gergely Szilvasy and Hugo Touvron and David Picard and Matthijs Douze and Hervé Jégou},
      year={2026},
      eprint={2611.09501},
      archivePrefix={arXiv},
      primaryClass={cs.LG},
      url={https://arxiv.org/abs/2611.09501}, 
}
```